# IOC Miner — Benchmark Evaluation

Evaluates ioc-miner extractors (regex, SecBERT NER, baselines) against the CyNER
ground truth dataset and produces P/R/F1 tables for the paper.

**Runtime:** CPU is fine — no GPU needed for this notebook.

**Sections:**
1. Setup — install deps, clone repo, mount Drive
2. Prepare eval dataset — CyNER → benchmark JSONL
3. Baseline comparison — regex vs iocextract vs ioc-finder
4. SecBERT NER benchmark — load fine-tuned model from Drive (optional)
5. Verdict accuracy — ContextClassifier evaluation
6. PDN case study — run pipeline on uploaded report files
7. Save results to Drive

## 1. Setup

In [1]:
# Clone repo and install ioc-miner
!git clone https://github.com/calvinkatoroy/ioc-miner.git
%cd ioc-miner

fatal: destination path 'ioc-miner' already exists and is not an empty directory.
/content/ioc-miner


In [2]:
%%capture
# Core dependencies (no torch needed for regex-only benchmark)
!pip install datasets transformers tokenizers
!pip install pdfplumber beautifulsoup4 lxml requests
!pip install nltk tldextract validators regex
!pip install rich typer

# Baseline tools
!pip install iocextract
!pip install ioc-finder

# Install ioc-miner itself in editable mode
!pip install -e . --quiet

In [3]:
import nltk
nltk.download('punkt_tab', quiet=True)
print('Setup complete.')

Setup complete.


In [4]:
# Mount Google Drive to persist results across sessions
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ioc-miner'
os.makedirs(f'{DRIVE_DIR}/eval',    exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('Drive mounted. Results will be saved to:', DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Results will be saved to: /content/drive/MyDrive/ioc-miner


## 2. Prepare Evaluation Dataset (CyNER → JSONL)

Downloads CyNER from HuggingFace and converts span annotations to ioc-miner's
benchmark JSONL format. Takes ~2 minutes on first run (downloads ~10 MB).

Output: `data/eval/cyner_eval.jsonl` — one sentence per line with IOC annotations.

In [5]:
!mkdir -p data/eval

!python scripts/prepare_eval_data.py \
    --source cyner \
    --split train \
    --output data/eval/cyner_eval.jsonl \
    --stats

INFO NumExpr defaulting to 2 threads.
INFO TensorFlow version 2.19.0 available.
INFO JAX version 0.7.2 available.
INFO Loading naorm/malware-text-db-cyner split=train ...
INFO HTTP Request: HEAD https://huggingface.co/datasets/naorm/malware-text-db-cyner/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/naorm/malware-text-db-cyner/716b340c8fbb8303d4b7a5c2e09cee987e835447/README.md "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/datasets/naorm/malware-text-db-cyner/resolve/716b340c8fbb8303d4b7a5c2e09cee987e835447/malware-text-db-cyner.py "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/naorm/malware-text-db-cyner/naorm/malware-text-db-cyner.py "HTTP/1.1 404 Not Found"
INFO HTTP Req

In [6]:
# Inspect a few samples
import json

with open('data/eval/cyner_eval.jsonl') as f:
    samples = [json.loads(l) for l in f if l.strip()][:5]

for s in samples:
    if s['iocs']:  # only show samples with IOCs
        print(f"Text : {s['text'][:100]}")
        for ioc in s['iocs']:
            print(f"  IOC: {ioc['type']:<10} {ioc['value']}")
        print()

## 3. Baseline Comparison

Runs ioc-miner regex extractor against `iocextract` and `ioc-finder`.

Expected runtime: 3–8 minutes depending on dataset size.

In [7]:
# Regex only — sanity check before baselines
!python scripts/run_benchmark.py \
    --eval data/eval/cyner_eval.jsonl \
    --extractors regex \
    --output results/regex_only.json

INFO Loading ground truth: data/eval/cyner_eval.jsonl
INFO Loaded 6733 sentences
INFO Loading extractor: regex

════════════════════════════════════════════════════════════════════════
  IOC EXTRACTION BENCHMARK
  Eval set : data/eval/cyner_eval.jsonl  (6733 sentences)
  Extractors: regex
════════════════════════════════════════════════════════════════════════

INFO Evaluating: regex
## regex

| IOC Type   | Precision | Recall | F1    | TP | FP | FN |
|------------|-----------|--------|-------|----|----|----|
| cve        | 0.429     | 0.094  | 0.154 | 33 | 44 | 319 |
| domain     | 0.416     | 0.898  | 0.569 | 274 | 384 | 31 |
| email      | 1.000     | 1.000  | 1.000 |  4 |  0 |  0 |
| filepath   | 0.000     | 0.000  | 0.000 |  0 | 213 |  0 |
| ip         | 0.418     | 0.903  | 0.571 | 28 | 39 |  3 |
| md5        | 0.340     | 1.000  | 0.507 | 17 | 33 |  0 |
| sha1       | 0.500     | 1.000  | 0.667 |  5 |  5 |  0 |
| sha256     | 0.000     | 0.000  | 0.000 |  0 |  6 |  0 |
| url    

In [8]:
# Full comparison: regex + iocextract + ioc-finder
# (cacador requires Go binary — skip for now)
!python scripts/run_benchmark.py \
    --eval data/eval/cyner_eval.jsonl \
    --extractors regex iocextract ioc-finder \
    --output results/baseline_comparison.json

INFO Loading ground truth: data/eval/cyner_eval.jsonl
INFO Loaded 6733 sentences
INFO Loading extractor: regex
INFO Loading extractor: iocextract
INFO Loading extractor: ioc-finder

════════════════════════════════════════════════════════════════════════
  IOC EXTRACTION BENCHMARK
  Eval set : data/eval/cyner_eval.jsonl  (6733 sentences)
  Extractors: regex, iocextract, ioc-finder
════════════════════════════════════════════════════════════════════════

INFO Evaluating: regex
INFO Evaluating: iocextract
WARNING Evaluation failed for iocextract: module 'iocextract' has no attribute 'extract_md5s'
INFO Evaluating: ioc-finder
| IOC Type   | regex P | ioc-finder P | regex R | ioc-finder R | regex F1 | ioc-finder F1 |
|------------|--------|--------|--------|--------|--------|--------|
| cve        | 0.429  | 0.418  | 0.094 | 0.094 | 0.154 | 0.153 |
| domain     | 0.416  | 0.316  | 0.898 | 0.311 | 0.569 | 0.314 |
| email      | 1.000  | 0.103  | 1.000 | 1.000 | 1.000 | 0.186 |
| filepath   

In [9]:
# Render results as a clean table for paper
import json
from ioc_miner.evaluation.benchmark import BenchmarkEvaluator, EvalResults, ExtractionMetrics
from ioc_miner.models.ioc import IOCType

with open('results/baseline_comparison.json') as f:
    raw = json.load(f)

# Reconstruct EvalResults objects
def _from_dict(d: dict) -> EvalResults:
    per_type = {}
    for k, v in d['per_type'].items():
        per_type[IOCType(k)] = ExtractionMetrics(tp=v['tp'], fp=v['fp'], fn=v['fn'])
    micro_d = d['micro']
    micro = ExtractionMetrics(tp=micro_d['tp'], fp=micro_d['fp'], fn=micro_d['fn'])
    return EvalResults(per_type=per_type, micro=micro)

comparison = {name: _from_dict(v) for name, v in raw['extraction'].items()}

print(BenchmarkEvaluator.comparison_markdown(comparison))

| IOC Type   | regex P | ioc-finder P | regex R | ioc-finder R | regex F1 | ioc-finder F1 |
|------------|--------|--------|--------|--------|--------|--------|
| cve        | 0.429  | 0.418  | 0.094 | 0.094 | 0.154 | 0.153 |
| domain     | 0.416  | 0.316  | 0.898 | 0.311 | 0.569 | 0.314 |
| email      | 1.000  | 0.103  | 1.000 | 1.000 | 1.000 | 0.186 |
| filepath   | 0.000  | 0.000  | 0.000 | 0.000 | 0.000 | 0.000 |
| ip         | 0.418  | 0.431  | 0.903 | 1.000 | 0.571 | 0.602 |
| md5        | 0.340  | 0.347  | 1.000 | 1.000 | 0.507 | 0.515 |
| sha1       | 0.500  | 0.500  | 1.000 | 1.000 | 0.667 | 0.667 |
| sha256     | 0.000  | 0.000  | 0.000 | 0.000 | 0.000 | 0.000 |
| url        | 0.500  | 0.026  | 1.000 | 1.000 | 0.667 | 0.050 |
| **micro**  | 0.333  | 0.313  | 0.506 | 0.260 | **0.402** | **0.284** |


## 4. SecBERT NER Benchmark (Optional)

Requires a fine-tuned model saved to Drive from `colab_finetune.ipynb`.

Also needs `torch` and `transformers` with a GPU for reasonable speed — switch to
**Runtime → T4 GPU** before running this section.

In [10]:
import os

MODEL_DIR = f'{DRIVE_DIR}/secbert-ioc-ner'

if os.path.exists(MODEL_DIR):
    print('Model found:', MODEL_DIR)
    print('Files:', os.listdir(MODEL_DIR))
else:
    print('Model NOT found at', MODEL_DIR)
    print('Run colab_finetune.ipynb first, then come back to this section.')

Model found: /content/drive/MyDrive/ioc-miner/secbert-ioc-ner
Files: ['checkpoint-170', 'checkpoint-340', 'checkpoint-510', 'checkpoint-680', 'checkpoint-850', 'config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'label_names.json', 'eval_results.json']


In [11]:
# Run only if model exists
import os
if os.path.exists(MODEL_DIR):
    !python scripts/run_benchmark.py \
        --eval data/eval/cyner_eval.jsonl \
        --extractors regex secbert iocextract ioc-finder \
        --model {MODEL_DIR} \
        --output results/full_comparison.json
else:
    print('Skipping — no fine-tuned model found.')

INFO Loading ground truth: data/eval/cyner_eval.jsonl
INFO Loaded 6733 sentences
INFO Loading extractor: regex
INFO Loading extractor: secbert
INFO Loading extractor: iocextract
INFO Loading extractor: ioc-finder

════════════════════════════════════════════════════════════════════════
  IOC EXTRACTION BENCHMARK
  Eval set : data/eval/cyner_eval.jsonl  (6733 sentences)
  Extractors: regex, secbert, iocextract, ioc-finder
════════════════════════════════════════════════════════════════════════

INFO Evaluating: regex
INFO Evaluating: secbert
WARNING Evaluation failed for secbert: 'NERExtractor' object has no attribute 'extract_all'
INFO Evaluating: iocextract
WARNING Evaluation failed for iocextract: module 'iocextract' has no attribute 'extract_md5s'
INFO Evaluating: ioc-finder
| IOC Type   | regex P | ioc-finder P | regex R | ioc-finder R | regex F1 | ioc-finder F1 |
|------------|--------|--------|--------|--------|--------|--------|
| cve        | 0.429  | 0.418  | 0.094 | 0.094 | 0

## 5. Verdict Accuracy (ContextClassifier)

Evaluates the dual-stage context classifier (rule-based → NLI fallback).

**Note:** CyNER does not include verdict labels (malicious/benign), so the default
JSONL has `"verdict": "unknown"` for all entries — verdict evaluation will show
0 evaluated IOCs. To run this section meaningfully:

1. Take a subset of `cyner_eval.jsonl`
2. Manually annotate `verdict` fields (e.g. `"malicious"` for C2 IPs)
3. Save as `data/eval/verdict_labeled.jsonl`
4. Run the cell below against that file

Alternatively, the PDN case study (Section 6) provides a natural verdict ground truth
from the BSSN advisory.

In [12]:
VERDICT_GT = 'data/eval/verdict_labeled.jsonl'

import os
if os.path.exists(VERDICT_GT):
    !python scripts/run_benchmark.py \
        --eval {VERDICT_GT} \
        --extractors regex \
        --verdict \
        --output results/verdict_eval.json
else:
    print(f'{VERDICT_GT} not found.')
    print('Create a verdict-labeled JSONL to run this section.')
    print()
    print('Example format:')
    print('{"text": "C2 beacon to 185.220.101.47", "iocs": [{"type": "ip", "value": "185.220.101.47", "verdict": "malicious"}]}')

data/eval/verdict_labeled.jsonl not found.
Create a verdict-labeled JSONL to run this section.

Example format:
{"text": "C2 beacon to 185.220.101.47", "iocs": [{"type": "ip", "value": "185.220.101.47", "verdict": "malicious"}]}


In [13]:
# Same as above but with NLI zero-shot classifier enabled
# Slower (~2s per IOC on CPU) but resolves more UNKNOWN cases
import os
if os.path.exists(VERDICT_GT):
    !python scripts/run_benchmark.py \
        --eval {VERDICT_GT} \
        --extractors regex \
        --verdict --use-ml \
        --output results/verdict_eval_nli.json
else:
    print('Skipping — verdict_labeled.jsonl not found.')

Skipping — verdict_labeled.jsonl not found.


## 6. PDN Case Study

Runs the full ioc-miner pipeline on PDN ransomware report files.

**Before running this section:**
1. Download the following public reports:
   - BSSN Brain Cipher advisory (PDF) — search `"ADVISORY-BRAIN-CIPHER BSSN 2024"`
   - SentinelOne / Recorded Future / Trend Micro blog posts (save as PDF or HTML)
2. Upload them to Colab using the cell below, or copy from Drive

The script deduplicates IOCs across all files and outputs a CSV + summary table.

In [14]:
# Option A: Upload files from your local machine
from google.colab import files
import os

os.makedirs('data/case_study/pdn', exist_ok=True)
print('Select your PDN report files to upload (.pdf / .html / .txt):')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/case_study/pdn/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'Saved: {dest}')

Select your PDN report files to upload (.pdf / .html / .txt):


KeyboardInterrupt: 

In [ ]:
# Option B: Copy from Drive (if already uploaded there)
import shutil, os

DRIVE_PDN = f'{DRIVE_DIR}/case_study/pdn'
os.makedirs('data/case_study/pdn', exist_ok=True)

if os.path.exists(DRIVE_PDN):
    for fname in os.listdir(DRIVE_PDN):
        shutil.copy(f'{DRIVE_PDN}/{fname}', f'data/case_study/pdn/{fname}')
        print(f'Copied: {fname}')
else:
    print(f'Drive path not found: {DRIVE_PDN}')
    print('Upload files there first, or use Option A above.')

In [ ]:
import os
pdn_files = [f for f in os.listdir('data/case_study/pdn') if not f.startswith('.')]

if not pdn_files:
    print('No files in data/case_study/pdn/ — upload reports first.')
else:
    print(f'Running on {len(pdn_files)} files:', pdn_files)
    !python scripts/case_study_pdn.py \
        --input data/case_study/pdn/ \
        --output results/pdn_iocs.csv

In [ ]:
# Preview extracted IOCs
import os
if os.path.exists('results/pdn_iocs.csv'):
    with open('results/pdn_iocs.csv') as f:
        lines = f.readlines()

    print(f'Total: {len(lines)-1} IOCs')
    print()
    print(''.join(lines[:20]))  # header + first 19
else:
    print('No output yet — run the case study cell above first.')

## 7. Save Results to Drive

In [ ]:
import shutil, os, glob

result_files = glob.glob('results/*.json') + glob.glob('results/*.csv')

for fpath in result_files:
    dest = f'{DRIVE_DIR}/results/{os.path.basename(fpath)}'
    shutil.copy(fpath, dest)
    print(f'Saved: {dest}')

# Also save the eval JSONL to Drive for future sessions
if os.path.exists('data/eval/cyner_eval.jsonl'):
    shutil.copy('data/eval/cyner_eval.jsonl', f'{DRIVE_DIR}/eval/cyner_eval.jsonl')
    print(f'Saved: {DRIVE_DIR}/eval/cyner_eval.jsonl')

print('\nAll results saved to Drive.')

In [ ]:
# Download results directly to your local machine
from google.colab import files
import glob

for fpath in glob.glob('results/*.json') + glob.glob('results/*.csv'):
    files.download(fpath)